In [28]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np

# Load the model
model = tf.keras.models.load_model("/content/best_model.h5")

# Get input shape from model (ignore batch dimension)
input_shape = model.input_shape[1:]  # e.g. (224, 224, 1) or (224, 224, 3)
img_height, img_width, img_channels = input_shape

# Class names
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Streamlit page config
st.set_page_config(page_title="Brain Tumor Detection", page_icon="🧠", layout="centered")

# Add a banner image from URL
st.image(
    "https://png.pngtree.com/thumb_back/fh260/background/20250417/pngtree-glowing-human-brain-image_17208331.jpg",
    caption="Brain MRI Example",
    use_column_width=True
)

st.title("🧠 Brain Tumor Detection App")
st.write("Upload an MRI scan and the model will predict the tumor type.")

# Upload image
uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Open the image
    image = Image.open(uploaded_file)

    # Force correct channels
    if img_channels == 1:
        image = image.convert("L")  # grayscale
    else:
        image = image.convert("RGB")  # RGB

    # Display uploaded image
    st.image(image, caption="Uploaded Image", use_column_width=True)

    # Preprocess image
    img = image.resize((img_width, img_height))
    img_array = np.array(img) / 255.0

    # Add missing dimensions if grayscale
    if img_channels == 1:
        img_array = np.expand_dims(img_array, axis=-1)  # (H,W,1)

    img_array = np.expand_dims(img_array, axis=0)  # (1,H,W,C)

    # Prediction
    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)[0]
    confidence = np.max(prediction)

    # Result
    result = f"🔍 Predicted: **{class_names[predicted_class]}** ({confidence*100:.2f}% confidence)"
    st.subheader("Result:")
    st.success(result)


Writing app.py


In [16]:
!pip install streamlit pyngrok

In [29]:
from pyngrok import ngrok

ngrok.set_auth_token("32n85CRlq11fbYTo9fvAc0RmKsQ_7PqFFBPeQNpMkCsnVBEAe")

In [30]:
!python app.py

2025-09-25 01:10:12.861631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758762612.886213    7437 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758762612.893444    7437 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758762612.911935    7437 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758762612.912003    7437 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758762612.912009    7437 computation_placer.cc:177] computation placer alr

In [31]:
!streamlit run app.py --server.port 8501 &> logs.txt &

In [32]:
!pkill -f ngrok


In [33]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("🌍 Public URL:", public_url)

🌍 Public URL: NgrokTunnel: "https://a35349a8b34b.ngrok-free.app" -> "http://localhost:8501"
